## 40_export_jobs  (OLD workspace) — Jobs Migration step 1

Exports the 5 practice job definitions + the internal practice-job-4b-target
(6 job objects total, Jobs API 2.1) to jobs_export.json, zips it, and
publishes for download — the lift-and-shift "job definition" that gets
recreated in the NEW workspace.


In [ ]:

dbutils.widgets.text("token", "", "Personal access token (old workspace)")

TOKEN = dbutils.widgets.get("token").strip()
assert TOKEN, "Paste a personal access token into the 'token' widget (Settings -> Developer -> Access tokens)"


In [ ]:

import json, os, shutil
import requests

try:
    HOST = dbutils.notebook.getContext().apiUrl().get().rstrip("/")
except Exception:
    dbutils.widgets.text("host", "", "Workspace URL, e.g. https://<your-free-edition-workspace>.cloud.databricks.com")
    HOST = dbutils.widgets.get("host").strip().rstrip("/")

S = requests.Session()
S.headers.update({"Authorization": f"Bearer {TOKEN}"})

def api(method, path, params=None):
    r = S.request(method, f"{HOST}{path}", params=params, timeout=60)
    if r.status_code >= 400:
        raise RuntimeError(f"{method} {path} -> {r.status_code}: {r.text[:500]}")
    return r.json()


In [ ]:

# List every job, keep only the practice ones, and fetch full settings.
job_summaries, page_token = [], None
while True:
    params = {"limit": 25}
    if page_token:
        params["page_token"] = page_token
    page = api("GET", "/api/2.1/jobs/list", params)
    job_summaries.extend(page.get("jobs", []))
    page_token = page.get("next_page_token")
    if not page_token:
        break

practice = [j for j in job_summaries if (j.get("settings", {}).get("name", "") or "").startswith("practice-job-")]
assert practice, "No practice-job-* jobs found — run 20_create_source_jobs first"

exports = []
for j in sorted(practice, key=lambda x: x["settings"]["name"]):
    detail = api("GET", "/api/2.1/jobs/get", {"job_id": j["job_id"]})
    exports.append({
        "name": detail["settings"]["name"],
        "source_job_id": j["job_id"],
        "settings": detail["settings"],
    })
    print(f"EXPORTED  {detail['settings']['name']}")

assert len(exports) == 6, (
    f"Expected 6 job objects (5 practice jobs + practice-job-4b-target), found {len(exports)} "
    f"— check 20_create_source_jobs"
)


In [ ]:

EXPORT_DIR = "/tmp/jobs_export"
shutil.rmtree(EXPORT_DIR, ignore_errors=True)
os.makedirs(EXPORT_DIR, exist_ok=True)

with open(os.path.join(EXPORT_DIR, "jobs_export.json"), "w") as f:
    json.dump(exports, f, indent=2, default=str)
print(f"Wrote {len(exports)} job definitions.")

ZIP_LOCAL = shutil.make_archive("/tmp/jobs_export", "zip", EXPORT_DIR)
print(f"Zip ready: {ZIP_LOCAL}")

user = dbutils.notebook.getContext().userName().get()
fname = os.path.basename(ZIP_LOCAL)
try:
    ws_dir = f"/Workspace/Users/{user}/migration_export"
    os.makedirs(ws_dir, exist_ok=True)
    shutil.copy(ZIP_LOCAL, ws_dir)
    print(f"\nDOWNLOAD from the workspace file browser:\n  {ws_dir}/{fname}\n(right-click -> Download)")
except Exception as e:
    print(f"Workspace copy failed ({e}); falling back to DBFS FileStore...")
    dbutils.fs.cp(f"file:{ZIP_LOCAL}", f"dbfs:/FileStore/migration_export/{fname}")
    print(f"Download from: https://<your-old-workspace-url>/files/migration_export/{fname}")
